# Potential and Laplacian of a Uniform Cylinder

This notebook visualizes the gravitational potential of an infinitely long cylinder with uniform density. The cylinder is parallel to the $z$-axis, so the potential depends only on $x$ and $y$. We use dimensionless units with $G=\lambda=R=1$, where $\lambda$ is the mass per unit length and $R$ is the cylinder radius.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

G = 1.0
line_mass = 1.0
R = 1.0
rho_0 = line_mass / (np.pi * R**2)

def cylinder_potential(x, y):
    r = np.hypot(x, y)
    phi = np.empty_like(r, dtype=float)
    inside = r <= R
    phi[inside] = G * line_mass * (r[inside]**2 / R**2 - 1)
    phi[~inside] = 2 * G * line_mass * np.log(r[~inside] / R)
    return phi

def cylinder_gravity(x, y):
    x, y = np.broadcast_arrays(np.asarray(x, dtype=float), np.asarray(y, dtype=float))
    r = np.hypot(x, y)
    factor = np.empty_like(r)
    inside = r <= R
    factor[inside] = -2 * G * line_mass / R**2
    factor[~inside] = -2 * G * line_mass / r[~inside]**2
    return factor * x, factor * y

def figure_directory():
    cwd = Path.cwd()
    if cwd.name == 'gravity_field':
        result = cwd / 'figures'
    elif cwd.name == '2_potential_fields':
        result = cwd / 'gravity_field' / 'figures'
    elif (cwd / '2_potential_fields').exists():
        result = cwd / '2_potential_fields' / 'gravity_field' / 'figures'
    else:
        result = cwd / 'book' / '2_potential_fields' / 'gravity_field' / 'figures'
    result.mkdir(parents=True, exist_ok=True)
    return result

figure_dir = figure_directory()

## Calculating the two-dimensional Laplacian

The potential is sampled on a regular grid. We then approximate $\partial^2\Phi/\partial x^2+\partial^2\Phi/\partial y^2$ using finite differences. Poisson's equation predicts $\nabla^2\Phi/(4\pi G\rho_0)=1$ inside the cylinder and $0$ outside it.

In [2]:
coordinates = np.linspace(-2.2 * R, 2.2 * R, 401)
X, Y = np.meshgrid(coordinates, coordinates)
Phi = cylinder_potential(X, Y)
spacing = coordinates[1] - coordinates[0]
dPhi_dx = np.gradient(Phi, spacing, axis=1)
dPhi_dy = np.gradient(Phi, spacing, axis=0)
laplacian = (
    np.gradient(dPhi_dx, spacing, axis=1)
    + np.gradient(dPhi_dy, spacing, axis=0)
)
normalized_laplacian = laplacian / (4 * np.pi * G * rho_0)

fig, (ax_phi, ax_lap) = plt.subplots(1, 2, figsize=(11.5, 4.9), constrained_layout=True)

potential_levels = np.linspace(Phi.min(), Phi.max(), 28)
potential_map = ax_phi.contourf(
    X / R, Y / R, Phi / (G * line_mass),
    levels=potential_levels / (G * line_mass), cmap='coolwarm'
)
ax_phi.contour(X / R, Y / R, Phi / (G * line_mass), levels=12, colors='white',
               linewidths=0.45, alpha=0.6)
arrow_coordinates = np.linspace(-2.0 * R, 2.0 * R, 17)
Xq, Yq = np.meshgrid(arrow_coordinates, arrow_coordinates)
gx, gy = cylinder_gravity(Xq, Yq)
ax_phi.quiver(
    Xq / R, Yq / R, gx / (G * line_mass / R), gy / (G * line_mass / R),
    color='#202020', scale=18, width=0.006, pivot='mid'
)
fig.colorbar(potential_map, ax=ax_phi, label=r'$\Phi/(G\lambda)$', shrink=0.84)

laplacian_map = ax_lap.contourf(
    X / R, Y / R, np.clip(normalized_laplacian, 0, 1),
    levels=np.linspace(0, 1, 21), cmap='viridis'
)
fig.colorbar(laplacian_map, ax=ax_lap,
             label=r'$\nabla^2\Phi/(4\pi G\rho_0)$', shrink=0.84)

theta = np.linspace(0, 2 * np.pi, 500)
for ax in (ax_phi, ax_lap):
    ax.plot(np.cos(theta), np.sin(theta), color='white', lw=1.6)
    ax.plot(np.cos(theta), np.sin(theta), color='0.15', lw=0.7, alpha=0.8)
    ax.set(xlim=(-2.2, 2.2), ylim=(-2.2, 2.2), xlabel='$x/R$', ylabel='$y/R$')
    ax.set_aspect('equal')

ax_phi.set_title('Potential and gravity field')
ax_lap.set_title('Laplacian reveals the source')
fig.savefig(figure_dir / 'uniform_cylinder_laplacian.png', dpi=200, bbox_inches='tight')
plt.close(fig)

```{figure} figures/uniform_cylinder_laplacian.png
:width: 100%

Potential, gravity field, and numerically calculated Laplacian of a uniform infinite cylinder.
```